# Выбор лидера (champion) на новом пайплайне

Сравнение моделей (LogReg, RandomForest, XGBoost, LightGBM, CatBoost) и ансамблей (Voting/Stacking) на актуальном target `tp_sl_1_05` с фичами из `outputs/features_selected_tp_sl_1_05.txt`. Выбор champion по качеству и net PnL (комиссия 0.1%).

## 1. Импорты и загрузка данных

In [ ]:
import sys, os, numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score
import joblib
import warnings
warnings.filterwarnings('ignore')

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

HAS_XGB = False
HAS_LGB = False
HAS_CAT = False
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    pass
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    pass
try:
    import catboost as cb
    HAS_CAT = True
except ImportError:
    pass

labeled_path = os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet')
feat_path = os.path.join(_root, 'outputs', 'features_selected_tp_sl_1_05.txt')
scaler_path = os.path.join(_root, 'models', 'scaler_tp_sl_1_05.joblib')
if not os.path.exists(labeled_path):
    raise FileNotFoundError(f'Не найден {labeled_path}. Запустите 04_Data_Labeling_And_Feature_Loading.ipynb')
if not os.path.exists(feat_path):
    raise FileNotFoundError(f'Не найден {feat_path}. Запустите 06_Feature_Selection.ipynb')
if not os.path.exists(scaler_path):
    raise FileNotFoundError(f'Не найден {scaler_path}. Запустите 07_Scaling_And_Normalization.ipynb')

df = pd.read_parquet(labeled_path)
with open(feat_path, encoding='utf-8') as f:
    FEATURES = [l.strip() for l in f if l.strip()]
FEATURES = [c for c in FEATURES if c in df.columns]
scaler_pack = joblib.load(scaler_path)
TARGET_COL = scaler_pack.get('target', 'target')
scaler = scaler_pack['scaler']

valid = df.dropna(subset=FEATURES + [TARGET_COL]).copy()
valid = valid[valid[TARGET_COL].isin([-1.0, 1.0])]
valid['y'] = (valid[TARGET_COL] == 1).astype(int)

sessions = valid['session_key'].unique().tolist()
np.random.seed(42)
np.random.shuffle(sessions)
n = len(sessions)
train_s = set(sessions[:int(0.7*n)])
val_s = set(sessions[int(0.7*n):int(0.85*n)])
test_s = set(sessions[int(0.85*n):])
train_df = valid[valid['session_key'].isin(train_s)].copy()
val_df = valid[valid['session_key'].isin(val_s)].copy()
test_df = valid[valid['session_key'].isin(test_s)].copy()

for _df in (train_df, val_df, test_df):
    sort_col = 'datetime' if 'datetime' in _df.columns else 'timestamp'
    _df.sort_values(['session_key', sort_col], inplace=True)
    _df['ret_next'] = _df.groupby('session_key')['close_price'].pct_change().shift(-1)

X_train = scaler.transform(train_df[FEATURES].fillna(0))
X_val = scaler.transform(val_df[FEATURES].fillna(0))
X_test = scaler.transform(test_df[FEATURES].fillna(0))
y_train = train_df['y'].values
y_val = val_df['y'].values
y_test = test_df['y'].values

print(f'Target: {TARGET_COL}, features: {len(FEATURES)}')
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
print(f'XGB: {HAS_XGB}, LGB: {HAS_LGB}, CatBoost: {HAS_CAT}')

## 2. Сложные базовые модели

In [ ]:
base_models = []

# Более сильные табличные базовые модели
base_models.append(('logreg', LogisticRegression(max_iter=1200, random_state=42)))
base_models.append(('rf', RandomForestClassifier(
    n_estimators=300, max_depth=14, min_samples_split=8,
    min_samples_leaf=4, max_features='sqrt', random_state=42
)))

if HAS_XGB:
    base_models.append(('xgb', xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, eval_metric='logloss'
    )))

if HAS_LGB:
    base_models.append(('lgb', lgb.LGBMClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, verbose=-1
    )))

if HAS_CAT:
    base_models.append(('cat', cb.CatBoostClassifier(
        iterations=300, depth=6, learning_rate=0.05,
        random_seed=42, verbose=0
    )))

print('Базовые модели:', [m[0] for m in base_models])

## 3. Обучение базовых моделей и метрики

In [ ]:
fitted = {}
rows = []

def best_threshold_by_f1(y_true, proba):
    best_thr, best_f1 = 0.5, -1.0
    for thr in np.round(np.arange(0.45, 0.71, 0.01), 2):
        f1 = f1_score(y_true, (proba >= thr).astype(int))
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)
    return best_thr, best_f1

for name, model in base_models:
    model.fit(X_train, y_train)
    fitted[name] = model
    p_val = model.predict_proba(X_val)[:, 1]
    p_test = model.predict_proba(X_test)[:, 1]
    thr, f1_val = best_threshold_by_f1(y_val, p_val)
    auc_val = roc_auc_score(y_val, p_val)
    auc_test = roc_auc_score(y_test, p_test)
    f1_test = f1_score(y_test, (p_test >= thr).astype(int))
    rows.append({
        'model': name,
        'auc_val': auc_val,
        'auc_test': auc_test,
        'thr': thr,
        'f1_val': f1_val,
        'f1_test_tuned': f1_test
    })

metrics_df = pd.DataFrame(rows).sort_values('auc_val', ascending=False)
print('Качество базовых моделей (выбор по val):')
print(metrics_df.to_string(index=False))

## 4. Ансамбль: Voting (soft) и Stacking

In [ ]:
ensemble_candidates = {}
if len(base_models) >= 2:
    voting = VotingClassifier(estimators=base_models, voting='soft', weights=None)
    voting.fit(X_train, y_train)
    ensemble_candidates['ensemble_voting'] = voting

    stacking = StackingClassifier(
        estimators=base_models,
        final_estimator=LogisticRegression(max_iter=800, random_state=42),
        cv=3
    )
    stacking.fit(X_train, y_train)
    ensemble_candidates['ensemble_stacking'] = stacking

    ens_rows = []
    for name, model in ensemble_candidates.items():
        p_val = model.predict_proba(X_val)[:, 1]
        p_test = model.predict_proba(X_test)[:, 1]
        thr, f1_val = best_threshold_by_f1(y_val, p_val)
        ens_rows.append({
            'model': name,
            'auc_val': roc_auc_score(y_val, p_val),
            'auc_test': roc_auc_score(y_test, p_test),
            'thr': thr,
            'f1_val': f1_val,
            'f1_test_tuned': f1_score(y_test, (p_test >= thr).astype(int))
        })
    ens_df = pd.DataFrame(ens_rows).sort_values('auc_val', ascending=False)
    print('Качество ансамблей (выбор по val):')
    print(ens_df.to_string(index=False))
else:
    ens_df = pd.DataFrame(columns=['model','auc_val','auc_test','thr','f1_val','f1_test_tuned'])
    print('Ансамбли пропущены: доступно <2 базовых моделей')

## 5. Бектест с комиссией 0.1%

In [ ]:
COMMISSION = 0.001

def backtest_pnl(proba, ret, commission=0.001, threshold=0.6):
    pred = np.where(proba >= threshold, 1, np.where(proba <= 1 - threshold, 0, -1))
    trade = np.where(pred == 1, 1, np.where(pred == 0, -1, 0))
    pnl_raw = trade * ret
    fee = commission * (trade != 0)
    pnl_net = pnl_raw - fee
    return {'trades': int((trade != 0).sum()), 'gross_%': pnl_raw.sum() * 100, 'net_%': pnl_net.sum() * 100}

bt = test_df.dropna(subset=['ret_next']).copy()
X_bt = scaler.transform(bt[FEATURES].fillna(0))
ret = bt['ret_next'].values

all_models = dict(fitted)
all_models.update(ensemble_candidates)

bt_rows = []
for _, mrow in pd.concat([metrics_df, ens_df], ignore_index=True).iterrows():
    name = mrow['model']
    thr = float(mrow['thr'])
    model = all_models[name]
    proba = model.predict_proba(X_bt)[:, 1]
    r_tuned = backtest_pnl(proba, ret, COMMISSION, threshold=thr)
    r_055 = backtest_pnl(proba, ret, COMMISSION, threshold=0.55)
    bt_rows.append({
        'model': name,
        'thr_tuned': thr,
        'net_tuned_%': r_tuned['net_%'],
        'trades_tuned': r_tuned['trades'],
        'net_055_%': r_055['net_%'],
        'trades_055': r_055['trades']
    })

bt_df = pd.DataFrame(bt_rows).sort_values('net_tuned_%', ascending=False)
print('Бектест (комиссия 0.1%, test):')
print(bt_df.to_string(index=False))

## 6. Итог: есть ли прибыль?

In [ ]:
leaderboard = pd.concat([metrics_df, ens_df], ignore_index=True).merge(bt_df, on='model', how='left')
leaderboard = leaderboard.sort_values(['net_tuned_%', 'auc_test'], ascending=False)

print('Единый leaderboard (качество + PnL):')
print(leaderboard.to_string(index=False))

# Champion: сначала прибыльность на test, затем AUC_test
champion_row = leaderboard.iloc[0]
champion_name = champion_row['model']
champion_thr = float(champion_row['thr'])
print(f"\nChampion: {champion_name} | net_tuned={champion_row['net_tuned_%']:.2f}% | auc_test={champion_row['auc_test']:.4f} | thr={champion_thr:.2f}")

## 7. Сохранение лучшей модели и ансамбля

In [ ]:
model_registry = dict(fitted)
model_registry.update(ensemble_candidates)
champion_model = model_registry[champion_name]

os.makedirs(os.path.join(_root, 'models'), exist_ok=True)
os.makedirs(os.path.join(_root, 'outputs'), exist_ok=True)

artifact = {
    'model': champion_model,
    'model_name': champion_name,
    'scaler': scaler,
    'features': FEATURES,
    'target': TARGET_COL,
    'threshold': champion_thr,
    'auc_val': float(champion_row['auc_val']),
    'auc_test': float(champion_row['auc_test']),
    'f1_test_tuned': float(champion_row['f1_test_tuned']),
    'net_tuned_%': float(champion_row['net_tuned_%']),
    'trades_tuned': int(champion_row['trades_tuned']),
    'split': {'train': 0.70, 'val': 0.15, 'test': 0.15, 'seed': 42},
    'source': '05_experiments/15_Complex_Models_And_Ensemble.ipynb'
}

champion_path = os.path.join(_root, 'models', 'champion_hackathon_tp_sl_1_05.joblib')
joblib.dump(artifact, champion_path)

leaderboard_path = os.path.join(_root, 'outputs', 'leaderboard_hackathon_tp_sl_1_05.csv')
leaderboard.to_csv(leaderboard_path, index=False)

print(f'Сохранён champion: {champion_path}')
print(f'Сохранён leaderboard: {leaderboard_path}')